# NoiseFloor - Notebook Addendum

**New analyses on top of the locked findings. This notebook does NOT modify any locked result**
(India vs China tone averages, t = -13.95, p = 0.0051, cooperative-coverage percentages all stay as published).

Run this in Google Colab **after** the main `NoiseFloor.ipynb` has built its parquet
(`noisefloor_pipeline/processed/nepal_events`), or re-download the windows it needs.

| Part | What it produces | Feeds into |
|------|------------------|------------|
| R1 | Nepal's tone **toward** India & China (reverse direction) | new CSV -> bundle/Power BI |
| R2 | All-neighbour x incident tone matrix | Trends heatmap upgrade |
| R3 | Weekly India/China aggregation | already bundled (`weekly_india_china`) |

**Factual anchor (professor's correction):** Sept 9 2025 = protests peak / parliament burned /
PM Oli resigned. **Sept 12 2025 = interim government under Sushila Karki sworn in.**
Keep these two dates distinct in every chart caption and annotation.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count
from pyspark.sql.functions import round as sround, lit

spark = SparkSession.builder.appName('NoiseFloor-Addendum').getOrCreate()

# Reuse the processed parquet from the main notebook.
# If starting a fresh session, re-run the main notebook's download + parse cells first.
nepal_events_df = spark.read.parquet('noisefloor_pipeline/processed/nepal_events')
print('events loaded:', nepal_events_df.count())

## R1 - Nepal's tone toward India & China (reverse direction)

The main pipeline measured **India/China -> Nepal** (`Actor1CountryCode` = IND/CN, Actor2 = NPL).
Here we flip it: Nepal as the *reporting* actor covering India and China
(`Actor1CountryCode == 'NPL'`, `Actor2CountryCode` IN/CHN). Same GDELT Events data,
same AvgTone field - a genuinely new comparison axis.

In [ ]:
nepal_reporting_df = nepal_events_df \
    .filter(col('Actor1CountryCode') == 'NPL') \
    .filter(col('Actor2CountryCode').isin('IND', 'CHN'))

reverse_daily_df = nepal_reporting_df \
    .groupBy('event_date', 'Actor2CountryCode') \
    .agg(
        sround(avg('AvgTone'), 4).alias('avg_tone'),
        count('GLOBALEVENTID').alias('event_count')) \
    .orderBy('event_date', 'Actor2CountryCode')

reverse_pivot_df = reverse_daily_df.groupBy('event_date').pivot('Actor2CountryCode') \
    .agg({'avg_tone': 'first', 'event_count': 'sum'})

import os
os.makedirs('noisefloor_pipeline/processed/powerbi_exports', exist_ok=True)
reverse_pivot_df.toPandas().to_csv(
    'noisefloor_pipeline/processed/powerbi_exports/nepal_toward_india_china.csv', index=False)

from google.colab import files
files.download('noisefloor_pipeline/processed/powerbi_exports/nepal_toward_india_china.csv')
reverse_pivot_df.orderBy('event_date', ascending=False).show(15, truncate=False)
print('Drop this CSV next to the others and rerun backend/build_data.py to bundle it.')

## R2 - All-neighbour x incident tone matrix

Gives the heatmap real per-incident granularity (today's dashboard heatmap uses overall
per-country averages from `source_countries.csv`). Windows match the main notebook:
- Earthquake: 2015-04-15 to 2015-06-15
- Blockade: 2015-09-20 to 2015-12-20
- Gen-Z protests: 2025-08-25 to 2025-11-15

In [ ]:
WINDOWS = {
    '2015 Earthquake': ('2015-04-15', '2015-06-15'),
    '2015 Blockade':   ('2015-09-20', '2015-12-20'),
    '2025 Gen-Z Protest': ('2025-08-25', '2025-11-15'),
}
NEIGHBOURS = ['IND','CHN','BTN','BGD','PAK','LKA','MDV','AFG','MMR']

frames = []
for period, (lo, hi) in WINDOWS.items():
    w = nepal_events_df.filter((col('event_date') >= lo) & (col('event_date') <= hi)) \
        .filter(col('Actor1CountryCode').isin(NEIGHBOURS)) \
        .groupBy('Actor1CountryCode') \
        .agg(sround(avg('AvgTone'), 4).alias('avg_tone'), count('GLOBALEVENTID').alias('events')) \
        .withColumn('period', lit(period))
    frames.append(w.toPandas())

import pandas as pd
matrix_pd = pd.concat(frames)
pivot_pd = matrix_pd.pivot(index='Actor1CountryCode', columns='period', values='avg_tone')
matrix_pd.to_csv('noisefloor_pipeline/processed/powerbi_exports/neighbour_matrix.csv', index=False)
files.download('noisefloor_pipeline/processed/powerbi_exports/neighbour_matrix.csv')
pivot_pd

## R3 - Weekly aggregation (already bundled)
`backend/build_data.py` derives `weekly_india_china` from `india_china_comparison.csv`,
so no extra export is required - this part is informational only.